# Facial Expression Recognition (FER2013) — PyTorch Experiments

Iterative CNN experiments on the Kaggle *Facial Expression Recognition Challenge*.

Approach: start small, add capacity, then regularize. Each architecture is trained with a few
**manual hyperparameter variations** (optimizer / learning rate / dropout), and every run is
logged to **Weights & Biases** as a separate run. We deliberately produce an **underfitting**,
an **overfitting**, and a **regularized** model and analyse what causes each.

In [1]:
!pip install kagglehub pandas numpy torch torchvision wandb

In [2]:
import pandas as pd

df = pd.read_csv('train.csv')

print("file successfully loaded")
print("dataset size:", df.shape)
print(df.head())

EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
print("\nclass distribution:")
for i, name in enumerate(EMOTIONS):
    print(f"  {i} {name}: {(df['emotion'] == i).sum()}")

file successfully loaded
dataset size: (28709, 2)
   emotion                                             pixels
0        0  70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1        0  151 150 147 155 148 133 111 140 170 174 182 15...
2        2  231 212 156 164 174 138 161 173 182 200 106 38...
3        4  24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4        6  4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...

class distribution:
  0 Angry: 3995
  1 Disgust: 436
  2 Fear: 4097
  3 Happy: 7215
  4 Sad: 4830
  5 Surprise: 3171
  6 Neutral: 4965


In [3]:
!pip install wandb
import wandb

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dgrig23 (dgrig23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import pandas as pd
import numpy as np

class FERDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        pixels = self.df.iloc[idx]['pixels']
        image = np.array(pixels.split(), dtype=np.uint8).reshape(48, 48)
        label = int(self.df.iloc[idx]['emotion'])
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

clean_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_dataset = FERDataset('train.csv', transform=clean_transform)

dataset_size = len(full_dataset)
train_size = int(0.85 * dataset_size)

torch.manual_seed(42)
indices = torch.randperm(dataset_size).tolist()

train_dataset = Subset(full_dataset, indices[:train_size])
val_dataset = Subset(full_dataset, indices[train_size:])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

Training images: 24402
Validation images: 4307


## Architectures

Three models, each defined once and parameterized so the same class can be reused across
hyperparameter runs (`dropout_rate` is read by the deep / optimized models).

In [5]:
import torch.nn as nn
import torch.nn.functional as F

class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(32 * 12 * 12, 7)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

class DeepOverfittedCNN(nn.Module):
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(256 * 6 * 6, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 7)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))
        x = self.pool(F.relu(self.conv4(x)))
        x = F.relu(self.conv5(x))
        x = self.pool(F.relu(self.conv6(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)
        return x

class OptimizedCNN(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)
        self.conv6 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(256 * 6 * 6, 1024)
        self.bn_fc1 = nn.BatchNorm1d(1024)
        self.fc2 = nn.Linear(1024, 7)

    def forward(self, x):
        x = self.pool(F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))))
        x = self.dropout(x)
        x = self.pool(F.relu(self.bn4(self.conv4(F.relu(self.bn3(self.conv3(x)))))))
        x = self.dropout(x)
        x = self.pool(F.relu(self.bn6(self.conv6(F.relu(self.bn5(self.conv5(x)))))))
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn_fc1(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [6]:
dummy = torch.randn(2, 1, 48, 48)
for m in [BaselineCNN(), DeepOverfittedCNN(dropout_rate=0.0), OptimizedCNN(dropout_rate=0.3)]:
    out = m(dummy)
    print(f"{m.__class__.__name__:20s} output shape: {tuple(out.shape)}")

BaselineCNN          output shape: (2, 7)
DeepOverfittedCNN    output shape: (2, 7)
OptimizedCNN         output shape: (2, 7)


In [7]:
import torch.optim as optim

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def sanity_overfit_single_batch(model, steps=200, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    images, labels = next(iter(train_loader))
    images, labels = images.to(device), labels.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    model.train()
    loss = None
    for _ in range(steps):
        opt.zero_grad()
        out = model(images)
        loss = crit(out, labels)
        loss.backward()
        opt.step()
    acc = (out.argmax(1) == labels).float().mean().item()
    print(f"Single-batch overfit -> final loss: {loss.item():.4f}, acc: {acc:.3f} (expected ~1.0)")
    return acc

In [8]:
sanity_overfit_single_batch(BaselineCNN(), steps=200, lr=1e-3)

Single-batch overfit -> final loss: 0.0005, acc: 1.000 (expected ~1.0)


1.0

In [9]:
import numpy as np
import wandb

ENTITY = "dgrig23-free-university-of-tbilisi-"
PROJECT = "facial-expression-recognition"
EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

def build_model(name, dropout_rate):
    if name == "baseline":
        return BaselineCNN()
    if name == "deep":
        return DeepOverfittedCNN(dropout_rate=dropout_rate)
    if name == "optimized":
        return OptimizedCNN(dropout_rate=dropout_rate)
    raise ValueError(f"unknown model: {name}")

def run_experiment(cfg):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    run = wandb.init(entity=ENTITY, project=PROJECT, name=cfg["name"], config=cfg, reinit=True)

    model = build_model(cfg["model"], cfg["dropout_rate"]).to(device)
    wandb.run.summary["num_parameters"] = count_parameters(model)
    wandb.watch(model, log="gradients", log_freq=100)

    criterion = nn.CrossEntropyLoss()
    if cfg["optimizer"] == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=cfg["lr"])
    else:
        optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])

    print(f"\n=== {cfg['name']} | params: {count_parameters(model):,} | device: {device} ===")

    best_val_acc = 0.0
    all_preds, all_labels = [], []

    for epoch in range(cfg["epochs"]):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = correct / total

        model.eval()
        val_running = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_running += loss.item() * images.size(0)
                _, pred = torch.max(outputs, 1)
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss = val_running / len(val_loader.dataset)
        val_acc = float((np.array(all_preds) == np.array(all_labels)).mean())
        best_val_acc = max(best_val_acc, val_acc)

        print(f"Epoch [{epoch+1}/{cfg['epochs']}] -> Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "train_val_gap": train_acc - val_acc,
            "learning_rate": optimizer.param_groups[0]['lr'],
        })

    wandb.run.summary["best_val_acc"] = best_val_acc

    wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
        probs=None, y_true=np.array(all_labels), preds=np.array(all_preds), class_names=EMOTIONS)})

    run.finish()
    print(f"[{cfg['name']}] best val acc: {best_val_acc:.4f}")
    return best_val_acc

## Architecture 1 — Baseline CNN

Two hyperparameter variations on the same small architecture, isolating the **optimizer / learning
rate** (this also keeps the later architecture comparison fair):

- `1a` SGD, lr=0.01
- `1b` Adam, lr=0.001

Expected: both **underfit** (low train and val accuracy), but Adam should learn faster.

In [10]:
baseline_experiments = [
    {"name": "1a_Baseline_SGD_lr0.01",   "model": "baseline", "optimizer": "SGD",  "lr": 0.01,  "dropout_rate": 0.0, "epochs": 10},
    {"name": "1b_Baseline_Adam_lr0.001", "model": "baseline", "optimizer": "Adam", "lr": 0.001, "dropout_rate": 0.0, "epochs": 10},
]
for cfg in baseline_experiments:
    run_experiment(cfg)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



=== 1a_Baseline_SGD_lr0.01 | params: 37,063 | device: cuda ===
Epoch [1/10] -> Train Loss: 1.7417, Train Acc: 0.2925 | Val Loss: 1.7068, Val Acc: 0.3192
Epoch [2/10] -> Train Loss: 1.6607, Train Acc: 0.3492 | Val Loss: 1.6531, Val Acc: 0.3480
Epoch [3/10] -> Train Loss: 1.6243, Train Acc: 0.3682 | Val Loss: 1.6282, Val Acc: 0.3536
Epoch [4/10] -> Train Loss: 1.5969, Train Acc: 0.3837 | Val Loss: 1.6234, Val Acc: 0.3566
Epoch [5/10] -> Train Loss: 1.5716, Train Acc: 0.3962 | Val Loss: 1.6238, Val Acc: 0.3531
Epoch [6/10] -> Train Loss: 1.5472, Train Acc: 0.4094 | Val Loss: 1.5855, Val Acc: 0.3877
Epoch [7/10] -> Train Loss: 1.5239, Train Acc: 0.4202 | Val Loss: 1.5717, Val Acc: 0.3984
Epoch [8/10] -> Train Loss: 1.5012, Train Acc: 0.4316 | Val Loss: 1.5186, Val Acc: 0.4314
Epoch [9/10] -> Train Loss: 1.4809, Train Acc: 0.4376 | Val Loss: 1.5325, Val Acc: 0.4086
Epoch [10/10] -> Train Loss: 1.4602, Train Acc: 0.4491 | Val Loss: 1.5162, Val Acc: 0.4182


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁
train_acc,▁▄▄▅▆▆▇▇▇█
train_loss,█▆▅▄▄▃▃▂▂▁
train_val_gap,▁▄▅▆█▆▆▄▇▇
val_acc,▁▃▃▃▃▅▆█▇▇
val_loss,█▆▅▅▅▄▃▁▂▁
best_val_acc,0.43139
epoch,10
learning_rate,0.01
num_parameters,37063


[1a_Baseline_SGD_lr0.01] best val acc: 0.4314



=== 1b_Baseline_Adam_lr0.001 | params: 37,063 | device: cuda ===
Epoch [1/10] -> Train Loss: 1.6096, Train Acc: 0.3688 | Val Loss: 1.5000, Val Acc: 0.4230
Epoch [2/10] -> Train Loss: 1.4170, Train Acc: 0.4662 | Val Loss: 1.4059, Val Acc: 0.4609
Epoch [3/10] -> Train Loss: 1.3230, Train Acc: 0.5061 | Val Loss: 1.3709, Val Acc: 0.4755
Epoch [4/10] -> Train Loss: 1.2488, Train Acc: 0.5324 | Val Loss: 1.3414, Val Acc: 0.4908
Epoch [5/10] -> Train Loss: 1.1922, Train Acc: 0.5612 | Val Loss: 1.3291, Val Acc: 0.4978
Epoch [6/10] -> Train Loss: 1.1409, Train Acc: 0.5800 | Val Loss: 1.3174, Val Acc: 0.5080
Epoch [7/10] -> Train Loss: 1.0987, Train Acc: 0.5961 | Val Loss: 1.3272, Val Acc: 0.5071
Epoch [8/10] -> Train Loss: 1.0571, Train Acc: 0.6137 | Val Loss: 1.3450, Val Acc: 0.5068
Epoch [9/10] -> Train Loss: 1.0177, Train Acc: 0.6286 | Val Loss: 1.3775, Val Acc: 0.4987
Epoch [10/10] -> Train Loss: 0.9865, Train Acc: 0.6390 | Val Loss: 1.4003, Val Acc: 0.5110


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁
train_acc,▁▄▅▅▆▆▇▇██
train_loss,█▆▅▄▃▃▂▂▁▁
train_val_gap,▁▃▄▅▅▆▆▇██
val_acc,▁▄▅▆▇███▇█
val_loss,█▄▃▂▁▁▁▂▃▄
best_val_acc,0.51103
epoch,10
learning_rate,0.001
num_parameters,37063


[1b_Baseline_Adam_lr0.001] best val acc: 0.5110


## Architecture 2 — Deep CNN (6 conv layers)

Large capacity, no BatchNorm. Two variations isolating **dropout**:

- `2a` dropout=0.0  -> expected to **overfit** hard (train accuracy near 1.0, val plateaus, val loss rises)
- `2b` dropout=0.5  -> same architecture, regularized -> the train/val gap should shrink

This pair is the core overfitting demonstration + its first fix.

In [11]:
deep_experiments = [
    {"name": "2a_Deep_dropout0.0_overfit", "model": "deep", "optimizer": "Adam", "lr": 0.001, "dropout_rate": 0.0, "epochs": 15},
    {"name": "2b_Deep_dropout0.5_reg",     "model": "deep", "optimizer": "Adam", "lr": 0.001, "dropout_rate": 0.5, "epochs": 15},
]
for cfg in deep_experiments:
    run_experiment(cfg)


=== 2a_Deep_dropout0.0_overfit | params: 11,110,855 | device: cuda ===
Epoch [1/15] -> Train Loss: 1.7646, Train Acc: 0.2667 | Val Loss: 1.6166, Val Acc: 0.3694
Epoch [2/15] -> Train Loss: 1.4952, Train Acc: 0.4185 | Val Loss: 1.3698, Val Acc: 0.4692
Epoch [3/15] -> Train Loss: 1.2758, Train Acc: 0.5088 | Val Loss: 1.2489, Val Acc: 0.5173
Epoch [4/15] -> Train Loss: 1.1191, Train Acc: 0.5706 | Val Loss: 1.2221, Val Acc: 0.5398
Epoch [5/15] -> Train Loss: 0.9809, Train Acc: 0.6290 | Val Loss: 1.1576, Val Acc: 0.5712
Epoch [6/15] -> Train Loss: 0.8212, Train Acc: 0.6951 | Val Loss: 1.1691, Val Acc: 0.5811
Epoch [7/15] -> Train Loss: 0.6281, Train Acc: 0.7707 | Val Loss: 1.3336, Val Acc: 0.5809
Epoch [8/15] -> Train Loss: 0.4317, Train Acc: 0.8473 | Val Loss: 1.6419, Val Acc: 0.5475
Epoch [9/15] -> Train Loss: 0.2859, Train Acc: 0.9001 | Val Loss: 1.9426, Val Acc: 0.5670
Epoch [10/15] -> Train Loss: 0.1952, Train Acc: 0.9334 | Val Loss: 2.1455, Val Acc: 0.5702
Epoch [11/15] -> Train Loss

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▃▄▅▅▆▇▇██████
train_loss,█▇▆▅▅▄▃▂▂▁▁▁▁▁▁
train_val_gap,▁▂▂▃▃▄▅▇▇▇█████
val_acc,▁▄▆▇███▇██▇▇█▇▇
val_loss,▃▂▁▁▁▁▂▃▄▅▇▇▆▇█
best_val_acc,0.58115
epoch,15
learning_rate,0.001
num_parameters,11110855


[2a_Deep_dropout0.0_overfit] best val acc: 0.5811



=== 2b_Deep_dropout0.5_reg | params: 11,110,855 | device: cuda ===
Epoch [1/15] -> Train Loss: 1.8302, Train Acc: 0.2422 | Val Loss: 1.8214, Val Acc: 0.2559
Epoch [2/15] -> Train Loss: 1.7504, Train Acc: 0.2785 | Val Loss: 1.6274, Val Acc: 0.3518
Epoch [3/15] -> Train Loss: 1.5386, Train Acc: 0.3994 | Val Loss: 1.4013, Val Acc: 0.4625
Epoch [4/15] -> Train Loss: 1.3535, Train Acc: 0.4796 | Val Loss: 1.2856, Val Acc: 0.5006
Epoch [5/15] -> Train Loss: 1.2513, Train Acc: 0.5198 | Val Loss: 1.2308, Val Acc: 0.5336
Epoch [6/15] -> Train Loss: 1.1591, Train Acc: 0.5561 | Val Loss: 1.2039, Val Acc: 0.5428
Epoch [7/15] -> Train Loss: 1.0842, Train Acc: 0.5909 | Val Loss: 1.1630, Val Acc: 0.5623
Epoch [8/15] -> Train Loss: 0.9913, Train Acc: 0.6284 | Val Loss: 1.1540, Val Acc: 0.5702
Epoch [9/15] -> Train Loss: 0.9060, Train Acc: 0.6605 | Val Loss: 1.1803, Val Acc: 0.5686
Epoch [10/15] -> Train Loss: 0.8111, Train Acc: 0.6982 | Val Loss: 1.2559, Val Acc: 0.5654
Epoch [11/15] -> Train Loss: 0.

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▁▃▄▄▅▅▆▆▆▇▇▇██
train_loss,██▇▆▅▅▄▄▃▃▂▂▂▁▁
train_val_gap,▂▁▁▂▂▃▃▄▄▅▆▆▇██
val_acc,▁▃▆▆▇▇█████████
val_loss,█▆▄▂▂▂▁▁▁▂▃▃▄▅▅
best_val_acc,0.57232
epoch,15
learning_rate,0.001
num_parameters,11110855


[2b_Deep_dropout0.5_reg] best val acc: 0.5723


## Architecture 3 — Optimized CNN (BatchNorm + Dropout)

Best architecture, tuned over **learning rate** and **dropout**:

- `3a` lr=0.001,  dropout=0.3 (reference)
- `3b` lr=0.0005, dropout=0.3 (lower LR)
- `3c` lr=0.001,  dropout=0.5 (stronger regularization)



In [12]:
optimized_experiments = [
    {"name": "3a_Optimized_lr0.001_do0.3",  "model": "optimized", "optimizer": "Adam", "lr": 0.001,  "dropout_rate": 0.3, "epochs": 15},
    {"name": "3b_Optimized_lr0.0005_do0.3", "model": "optimized", "optimizer": "Adam", "lr": 0.0005, "dropout_rate": 0.3, "epochs": 15},
    {"name": "3c_Optimized_lr0.001_do0.5",  "model": "optimized", "optimizer": "Adam", "lr": 0.001,  "dropout_rate": 0.5, "epochs": 15},
]
for cfg in optimized_experiments:
    run_experiment(cfg)


=== 3a_Optimized_lr0.001_do0.3 | params: 10,593,479 | device: cuda ===
Epoch [1/15] -> Train Loss: 1.5290, Train Acc: 0.4137 | Val Loss: 1.3237, Val Acc: 0.5020
Epoch [2/15] -> Train Loss: 1.2361, Train Acc: 0.5309 | Val Loss: 1.1771, Val Acc: 0.5493
Epoch [3/15] -> Train Loss: 1.1298, Train Acc: 0.5727 | Val Loss: 1.1207, Val Acc: 0.5681
Epoch [4/15] -> Train Loss: 1.0434, Train Acc: 0.6046 | Val Loss: 1.0866, Val Acc: 0.5897
Epoch [5/15] -> Train Loss: 0.9728, Train Acc: 0.6337 | Val Loss: 1.1050, Val Acc: 0.5883
Epoch [6/15] -> Train Loss: 0.9021, Train Acc: 0.6616 | Val Loss: 1.0334, Val Acc: 0.6164
Epoch [7/15] -> Train Loss: 0.8332, Train Acc: 0.6870 | Val Loss: 1.0956, Val Acc: 0.6051
Epoch [8/15] -> Train Loss: 0.7566, Train Acc: 0.7205 | Val Loss: 1.0778, Val Acc: 0.6130
Epoch [9/15] -> Train Loss: 0.6803, Train Acc: 0.7508 | Val Loss: 1.0698, Val Acc: 0.6264
Epoch [10/15] -> Train Loss: 0.6087, Train Acc: 0.7742 | Val Loss: 1.1356, Val Acc: 0.6239
Epoch [11/15] -> Train Loss

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▃▄▄▅▅▅▆▆▇▇▇██
train_loss,█▆▆▅▅▄▄▄▃▃▂▂▂▁▁
train_val_gap,▁▂▃▃▄▄▄▅▅▆▆▇▇██
val_acc,▁▃▄▆▅▇▆▇▇▇▇█▇██
val_loss,█▄▃▂▃▁▂▂▂▃▄▄▅▆█
best_val_acc,0.6371
epoch,15
learning_rate,0.001
num_parameters,10593479


[3a_Optimized_lr0.001_do0.3] best val acc: 0.6371



=== 3b_Optimized_lr0.0005_do0.3 | params: 10,593,479 | device: cuda ===
Epoch [1/15] -> Train Loss: 1.5582, Train Acc: 0.3961 | Val Loss: 1.3430, Val Acc: 0.4869
Epoch [2/15] -> Train Loss: 1.2526, Train Acc: 0.5235 | Val Loss: 1.1710, Val Acc: 0.5489
Epoch [3/15] -> Train Loss: 1.1394, Train Acc: 0.5697 | Val Loss: 1.1240, Val Acc: 0.5661
Epoch [4/15] -> Train Loss: 1.0470, Train Acc: 0.6026 | Val Loss: 1.0864, Val Acc: 0.5911
Epoch [5/15] -> Train Loss: 0.9816, Train Acc: 0.6292 | Val Loss: 1.0801, Val Acc: 0.5986
Epoch [6/15] -> Train Loss: 0.9165, Train Acc: 0.6582 | Val Loss: 1.0712, Val Acc: 0.6053
Epoch [7/15] -> Train Loss: 0.8484, Train Acc: 0.6840 | Val Loss: 1.1075, Val Acc: 0.5925
Epoch [8/15] -> Train Loss: 0.7896, Train Acc: 0.7076 | Val Loss: 1.0450, Val Acc: 0.6229
Epoch [9/15] -> Train Loss: 0.7262, Train Acc: 0.7324 | Val Loss: 1.0846, Val Acc: 0.6150
Epoch [10/15] -> Train Loss: 0.6550, Train Acc: 0.7595 | Val Loss: 1.0401, Val Acc: 0.6322
Epoch [11/15] -> Train Los

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▄▄▅▅▆▆▆▇▇▇██
train_loss,█▆▆▅▅▄▄▃▃▃▂▂▂▁▁
train_val_gap,▁▂▃▃▄▄▅▅▆▆▆▇▇██
val_acc,▁▄▅▆▆▆▆▇▇██████
val_loss,█▄▃▂▂▂▃▁▂▁▂▂▃▄▅
best_val_acc,0.64337
epoch,15
learning_rate,0.0005
num_parameters,10593479


[3b_Optimized_lr0.0005_do0.3] best val acc: 0.6434



=== 3c_Optimized_lr0.001_do0.5 | params: 10,593,479 | device: cuda ===
Epoch [1/15] -> Train Loss: 1.6503, Train Acc: 0.3627 | Val Loss: 1.3570, Val Acc: 0.4704
Epoch [2/15] -> Train Loss: 1.3539, Train Acc: 0.4789 | Val Loss: 1.2657, Val Acc: 0.5205
Epoch [3/15] -> Train Loss: 1.2488, Train Acc: 0.5211 | Val Loss: 1.2082, Val Acc: 0.5356
Epoch [4/15] -> Train Loss: 1.1820, Train Acc: 0.5514 | Val Loss: 1.1375, Val Acc: 0.5728
Epoch [5/15] -> Train Loss: 1.1346, Train Acc: 0.5742 | Val Loss: 1.1056, Val Acc: 0.5726
Epoch [6/15] -> Train Loss: 1.0876, Train Acc: 0.5861 | Val Loss: 1.0673, Val Acc: 0.5974
Epoch [7/15] -> Train Loss: 1.0484, Train Acc: 0.6019 | Val Loss: 1.0705, Val Acc: 0.5948
Epoch [8/15] -> Train Loss: 1.0129, Train Acc: 0.6154 | Val Loss: 1.0294, Val Acc: 0.6137
Epoch [9/15] -> Train Loss: 0.9685, Train Acc: 0.6347 | Val Loss: 1.0202, Val Acc: 0.6197
Epoch [10/15] -> Train Loss: 0.9421, Train Acc: 0.6427 | Val Loss: 1.0061, Val Acc: 0.6241
Epoch [11/15] -> Train Loss

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▅▅▆▆▆▇▇▇▇██
train_loss,█▆▅▄▄▄▃▃▃▂▂▂▂▁▁
train_val_gap,▁▃▄▄▅▅▅▅▆▆▆▇▇▇█
val_acc,▁▃▄▅▅▆▆▇▇▇▇████
val_loss,█▆▅▄▃▂▂▂▁▁▁▁▁▁▂
best_val_acc,0.63826
epoch,15
learning_rate,0.001
num_parameters,10593479


[3c_Optimized_lr0.001_do0.5] best val acc: 0.6383




- **Baseline (1a/1b)** -> low train AND val accuracy = **underfitting** (high bias).
- **Deep 2a** -> train accuracy near 1.0, val much lower, `train_val_gap` large and rising val loss
  = **overfitting** (high variance). **2b** (dropout) narrows that gap.
- **Optimized (3a-3c)** -> highest val accuracy, smallest gap = best **generalization**.


